In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [ ]:
from google.cloud import bigquery
import pandas as pd

# Inicializa el cliente de BigQuery
client = bigquery.Client(project='dataton-2024-team-01-cofares')

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas desde BigQuery datos_no_descriptions_eans
#query = "SELECT * FROM `dataton-2024-team-01-cofares.datos_cofares.datos_no_descriptions_eans`"
#df = client.query(query).to_dataframe()
#print(df.head())

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas
df = pd.read_csv("productos_no_description_df.csv")
print(df.head())

In [2]:
#EXTRAE SOLO 1 RESULTADO(el primero)

query = "2104792"
country = "ES"

def google_custom_search_df(query, country):
    API_KEY = os.getenv('API_KEY')
    API_CUSTOM_SEARCH_ID = os.getenv('API_CUSTOM_SEARCH_ID')

    if not API_KEY or not API_CUSTOM_SEARCH_ID:
        print("Error: API Key o ID de búsqueda no están cargados correctamente")
        return pd.DataFrame()

    # URL request para obtener solo los primeros 10 resultados
    url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY}&cx={API_CUSTOM_SEARCH_ID}&q={query}&start=1&gl={country}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error en la petición: {response.status_code}")
        return pd.DataFrame()

    data = response.json()

# Extraer solo el primer resultado
    items = data.get("items", [])
    if items:
        items = items[:1]  # Solo tomar el primer resultado

    if not items:
        print("No se encontraron resultados")
        return pd.DataFrame()

    # Crear un DataFrame con los resultados de la búsqueda
    df = pd.DataFrame(items, columns=['title', 'link']) 

    return df

# Llamada a la función para obtener solo los primeros 10 resultados
df_productos = google_custom_search_df(query, country)
print(df_productos)

                                               title  \
0  Comprar Paranix Loción Elimina Piojos Y Liendr...   

                                                link  
0  https://www.welnia.com/piojos/paranix-locion-e...  


In [3]:
#-------------------SCRAPING del texto producto

#Extraer el texto de los articulos con la libreria newspapper y, en su defecto, con beautiful soup
from bs4 import BeautifulSoup
from newspaper import Article

def scrape_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        if not article.text:
            # Si Newspaper no pudo obtener el texto, intenta con BeautifulSoup
            page = requests.get(url)
            soup = BeautifulSoup(page.content, 'html.parser')
            
            # Busca contenido en etiquetas <p> o <div>
            article_text = ' '.join([p.get_text() for p in soup.find_all(['p', 'div'])])
            
            return article_text
        return article.text
    except Exception as e:
        print(f"Error al scrapear el artículo: {str(e)}")
        return ""

def scrape_articles_in_dataframe(df):
    scraped_texts = []  # Aquí almacenaremos el texto de los artículos

# Solo procesar el primer enlace
    if not df.empty:
        url = df['link'].iloc[0]  # Obtener solo el primer enlace
        scraped_text = scrape_article(url)
        scraped_texts.append(scraped_text)

    # Agregamos los textos como una nueva columna en el DataFrame
    df['scraped_text'] = scraped_texts

    return df

# Llama a la función scraping de los artículos
df = scrape_articles_in_dataframe(df_productos)
print(df)

                                               title  \
0  Comprar Paranix Loción Elimina Piojos Y Liendr...   

                                                link  \
0  https://www.welnia.com/piojos/paranix-locion-e...   

                                        scraped_text  
0  Descripción\n\nParanix Loción Elimina Piojos y...  


In [ ]:
def extraer_ean(valor):
    # Si el valor es una cadena (string)
    if isinstance(valor, str):
        # Busca el EAN13 después de la comilla simple
        import re
        match = re.search(r"'EAN13': '(\d+)'", valor)
        if match:
            return match.group(1)
    
    # Si el valor es una lista de diccionarios
    elif isinstance(valor, list):
        if valor and isinstance(valor[0], dict):
            return valor[0].get('EAN13')
    
    return None

# Aplicar la función a la columna
df['ean_limpio'] = df['eans'].apply(extraer_ean)
print(df.head())

In [6]:
import vertexai
from vertexai import generative_models as genai
from vertexai.generative_models import GenerationConfig

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")


# Model definition
multimodal_model = genai.GenerativeModel( "gemini-1.5-flash",
generation_config= GenerationConfig(temperature=0))

In [5]:
def descripcion_producto_con_gemini(df):
    # Iterar sobre el DataFrame para procesar cada producto
    for index, row in df.iterrows():
        title = row['title']
        scraped_text = row['scraped_text']
        prompt = f"""
        Genera una descripción precisa del producto '{title}' basada en su texto scrapeado: {scraped_text}.
        Destaca las características clave del producto y asegúrate de que la descripción sea clara y concisa.
        Idioma: español. Codificación: UTF-8
        """
        try:
            response = multimodal_model.generate_content(prompt)
            descripcion = response.text
            print(f"Descripción de '{title}' generada con éxito.")
            return descripcion
        except Exception as e:
            print(f"Error al generar la descripción de '{title}' con Gemini: {str(e)}")
            return f"Error al generar la descripción de '{title}' con Gemini"

# Llamar a la función para generar la descripción
descripcion_producto = descripcion_producto_con_gemini(df)

# Guardar la descripción en un archivo CSV
import csv
with open("descripcion_producto.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([descripcion_producto])

print("\nDescripción del producto guardada en 'descripcion_producto.csv'")

Descripción de 'Comprar Iraltone DS Champú, 200 ml | Welnia' generada con éxito.

Descripción del producto guardada en 'descripcion_producto.csv'
